In [ ]:
import torch
import numpy as np
from tensordict import TensorDict
import matplotlib.pyplot as plt

from job_shop_lib import load_instance

# ===============================
# CONFIG
# ===============================

CHECKPOINT_PATH = "deepaco_jssp_6x6.pt"
EMBED_DIM = 64
NUM_EVALS = 50
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Using device:", DEVICE)

# ===============================
# LOAD FT06
# ===============================

instance = load_instance("ft06")

num_jobs = instance.num_jobs
num_machines = instance.num_machines

print(f"Loaded FT06: {num_jobs} jobs, {num_machines} machines")

# Extract machine order + processing times
machine_orders = []
proc_times_list = []

for job in instance.jobs:
    machines = []
    times = []
    for op in job.operations:
        machines.append(op.machine)
        times.append(op.duration)
    machine_orders.append(machines)
    proc_times_list.append(times)

machine_orders = np.array(machine_orders)
proc_times_array = np.array(proc_times_list)

# ===============================
# CONVERT TO TENSORDICT
# ===============================

# Number of operations
n_ops = num_jobs * num_machines

# Build processing tensor: (machines, ops)
proc_tensor = torch.zeros(num_machines, n_ops)

ops_job_map = []
start_op_per_job = []
end_op_per_job = []

op_counter = 0

for j in range(num_jobs):
    start_op_per_job.append(op_counter)
    for k in range(num_machines):
        m = machine_orders[j, k]
        t = proc_times_array[j, k]
        proc_tensor[m, op_counter] = t
        ops_job_map.append(j)
        op_counter += 1
    end_op_per_job.append(op_counter - 1)

pad_mask = torch.zeros(n_ops).bool()

td = TensorDict({
    "proc_times": proc_tensor.unsqueeze(0),  # batch dim
    "start_op_per_job": torch.tensor(start_op_per_job).unsqueeze(0),
    "end_op_per_job": torch.tensor(end_op_per_job).unsqueeze(0),
    "pad_mask": pad_mask.unsqueeze(0),
}, batch_size=[1]).to(DEVICE)

# ===============================
# REBUILD MODEL
# ===============================

generator = MyJSSPGenerator(
    num_jobs=num_jobs,
    num_machines=num_machines
)

env = OperationSelectionEnv(generator)

init_emb = JSSPInitEmbedding(
    embed_dim=EMBED_DIM,
    num_machines=num_machines
)

edge_emb = JsspEdgeEmbedding(embed_dim=EMBED_DIM)

encoder = NARGNNEncoder(
    embed_dim=EMBED_DIM,
    init_embedding=init_emb,
    edge_embedding=edge_emb
)

policy = DeepACOPolicy(
    encoder=encoder,
    env_name="tsp",
    n_ants=dict(train=10, val=20, test=50),
    n_iterations=dict(train=1, val=5, test=10),
    aco_class=MyAntSystem
)

model = DeepACO(
    env=env,
    policy=policy,
    train_with_local_search=False,
).to(DEVICE)

state_dict = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
model.load_state_dict(state_dict)
model.eval()

print("Model loaded successfully.")

# ===============================
# RUN EVALUATIONS
# ===============================

makespans = []

with torch.no_grad():
    for i in range(NUM_EVALS):
        out = model(td.clone(), phase="test")
        makespan = -out["reward"].item()
        makespans.append(makespan)
        print(f"Eval {i+1}/{NUM_EVALS} | Makespan: {makespan}")

print("\nBest makespan:", min(makespans))
print("Average makespan:", np.mean(makespans))

# ===============================
# PLOT
# ===============================

plt.plot(range(1, NUM_EVALS+1), makespans)
plt.xlabel("Evaluation")
plt.ylabel("Makespan")
plt.title("DeepACO on FT06")
plt.grid(True)
plt.show()